[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajaramillola1985/cauca-clima-gee/blob/main/notebooks/06_aptitud_riesgo_cafe.ipynb)

# 06 · Aptitud para café y riesgo de roya y muerte descendente (Phoma) en el Cauca

Genera para el departamento del Cauca:

1. **Aptitud para café** (multicriterio de clima, suelo y relieve).
2. **Riesgo de roya** (*Hemileia vastatrix*) y **de muerte descendente** (*Phoma* spp.), calculados sobre el **área sembrada real** de café.
3. **Análisis de sensibilidad a los pesos** (uno a la vez y Monte Carlo).
4. Mapas en PNG y GeoTIFF coloreados.

Cambios frente a la versión anterior:

- La temperatura y la amplitud térmica vienen de **TerraClimate 1995–2024**, corregidas por altitud con SRTM, en lugar de WorldClim v1 (1960–1990).
- El riesgo se calcula sobre el área sembrada (mapa de probabilidad de café de Forest Data Partnership, o tu propio asset), no sobre una zona modelada.
- Se agregó el análisis de sensibilidad.

El riesgo **mensual** y las alertas están en el notebook `07_riesgo_mensual_alertas.ipynb`.

Ejecuta las celdas en orden. Todo lo que se puede ajustar está en la sección 2.

## 1. Conexión a Earth Engine

In [ ]:
import ee, geemap
from ipyleaflet import basemaps

PROYECTO_GEE = 'sig-unal-2026'   # <-- tu ID de proyecto de Google Cloud, entre comillas

# Autenticación con enlace y código (evita el error "credential propagation was unsuccessful").
# Al generar el token, NO marques la opción de solo lectura: exportar necesita permisos completos.
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project=PROYECTO_GEE)

## 2. Configuración y criterios

- `trapecio: [a, b, c, d]`: puntaje 0 por debajo de `a`, sube a 1 en `b`, se mantiene hasta `c` y baja a 0 en `d`.
- `sube: [a, b]`: 0 en `a` o menos, 1 en `b` o más. `baja: [a, b]`: 1 en `a` o menos, 0 en `b` o más.
- Los pesos de cada índice deben sumar 1.

**Importante:** los umbrales de lluvia, humedad y amplitud se ajustaron con la climatología anterior y la zona modelada. Después de la primera corrida, revisa la tabla de la sección 10 y ajústalos si quedan saturados.

In [ ]:
# --- Periodo de la climatología (TerraClimate llega hasta 2024)
PERIODO_CLIMA = ('1995-01-01', '2025-01-01')

# --- Corrección de temperatura por altitud (°C por metro). Típico en los Andes: -0,0055 a -0,0065
GRADIENTE_TERMICO = -0.006

# --- Área sembrada de café
#   'fdp'   : probabilidad de café de Forest Data Partnership (10 m), umbral configurable
#   'asset' : tus propios polígonos de lotes o veredas cafeteras subidos como asset de Earth Engine
#   'modelo': la zona cafetera potencial de la versión anterior (aptitud >= 0,4 y 1.000-2.200 m)
FUENTE_AREA_CAFE = 'fdp'
ANIO_CAFE = 2024                 # FDP model_2026a cubre 2018-2024
UMBRAL_PROB_CAFE = 0.5           # súbelo (0,6-0,7) si sobreestima, bájalo si subestima
ASSET_AREA_CAFE = 'projects/sig-unal-2026/assets/area_cafe_cauca'   # solo si usas 'asset'

APTITUD_CAFE = {
    'temperatura': {'trapecio': [16, 18, 22, 24],         'peso': 0.30},  # °C, media anual
    'lluvia':      {'trapecio': [1200, 1500, 2800, 3500], 'peso': 0.20},  # mm/año
    'altitud':     {'trapecio': [1000, 1300, 1900, 2200], 'peso': 0.15},  # m
    'pendiente':   {'baja':     [50, 100],                'peso': 0.15},  # %
    'ph':          {'trapecio': [4.5, 5.0, 5.5, 6.5],     'peso': 0.20},  # pH
}

RIESGO_ROYA = {
    'temperatura': {'trapecio': [15, 21, 25, 28], 'peso': 0.45},  # °C
    'altitud':     {'baja':     [1300, 2000],     'peso': 0.30},  # m
    'lluvia':      {'sube':     [1500, 2500],     'peso': 0.15},  # mm/año
    'humedad':     {'sube':     [78, 92],         'peso': 0.10},  # %
}

RIESGO_PHOMA = {
    'altitud':     {'sube': [1400, 1900], 'peso': 0.35},  # m
    'temperatura': {'baja': [17, 21],     'peso': 0.25},  # °C
    'amplitud':    {'sube': [9.5, 11.5],  'peso': 0.20},  # °C día-noche
    'lluvia':      {'sube': [1800, 3200], 'peso': 0.20},  # mm/año
}

# Clases de ESA WorldCover excluidas de la aptitud: construido, nieve, agua, humedal, manglar
COBERTURAS_EXCLUIDAS = [50, 70, 80, 90, 95]
EXCLUIR_AREAS_PROTEGIDAS = True

# --- Análisis de sensibilidad
SENS_CAMBIOS = [-0.2, 0.2]   # variación relativa de cada peso (uno a la vez)
SENS_N_SIM = 30              # simulaciones Monte Carlo por índice
SENS_CONCENTRACION = 50      # más alto = pesos más cercanos a los base (con 50, un peso de 0,45 varía aprox. entre 0,30 y 0,55)
SENS_ESCALA = 250            # m; resolución para resumir la sensibilidad
SEMILLA = 42

for nombre, crit in [('aptitud', APTITUD_CAFE), ('roya', RIESGO_ROYA), ('phoma', RIESGO_PHOMA)]:
    total = sum(v['peso'] for v in crit.values())
    assert abs(total - 1) < 1e-6, f'Los pesos de {nombre} suman {total}, deben sumar 1'

## 3. Área de estudio

In [ ]:
cauca = (ee.FeatureCollection('FAO/GAUL/2015/level1')
         .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
         .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))   # nombre exacto: excluye Valle del Cauca
region = cauca.geometry()

municipios = (ee.FeatureCollection('FAO/GAUL/2015/level2')
              .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
              .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))

print('Departamentos encontrados:', cauca.size().getInfo())   # debe ser 1
print('Municipios:', municipios.size().getInfo())            # GAUL 2015 puede no dar 42

## 4. Funciones

In [ ]:
def puntaje(img, regla):
    """Convierte una capa a un puntaje de 0 a 1 según la regla del criterio."""
    if 'trapecio' in regla:
        a, b, c, d = regla['trapecio']
        sube = img.subtract(a).divide(b - a)
        baja = ee.Image.constant(d).subtract(img).divide(d - c)
        return sube.min(baja).clamp(0, 1)
    if 'sube' in regla:
        a, b = regla['sube']
        return img.subtract(a).divide(b - a).clamp(0, 1)
    if 'baja' in regla:
        a, b = regla['baja']
        return ee.Image.constant(b).subtract(img).divide(b - a).clamp(0, 1)
    raise ValueError(f'Regla no reconocida: {regla}')

def pesos_base(criterios):
    return {v: r['peso'] for v, r in criterios.items()}

def indice(criterios, nombre, img=None, pesos=None):
    """Suma ponderada de los puntajes (0 a 1). img: imagen con las variables; pesos: opcional."""
    img = capas if img is None else img
    pesos = pesos_base(criterios) if pesos is None else pesos
    total = ee.Image.constant(0)
    for variable, regla in criterios.items():
        total = total.add(puntaje(img.select(variable), regla).multiply(pesos[variable]))
    return total.rename(nombre).clip(region)

def clasificar(img):
    """Clases 1 a 5 con cortes en 0,2 / 0,4 / 0,6 / 0,8."""
    return (img.gte(0.2).add(img.gte(0.4)).add(img.gte(0.6)).add(img.gte(0.8))
            .add(1).rename('clase').updateMask(img.mask()))

def humedad_relativa(t, td):
    """Humedad relativa (%) con la aproximación de Magnus; t y td en °C."""
    pv = lambda x: x.multiply(17.625).divide(x.add(243.04)).exp()
    return pv(td).divide(pv(t)).multiply(100)

def hex_a_rgba(h):
    h = h.lstrip('#')
    return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), 255)

PALETA_APT = ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
ETIQ_APT = ['No apto', 'Marginal', 'Moderado', 'Apto', 'Muy apto']
PALETA_RIESGO = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
ETIQ_RIESGO = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
COLOR_FUERA = '#d9d9d9'

## 5. Capas de clima, suelo y relieve

La temperatura media de TerraClimate (~4,6 km) se lleva a 30 m con el gradiente térmico: se suma `GRADIENTE_TERMICO × (altitud SRTM − altitud media de la celda de TerraClimate)`. La amplitud térmica es el promedio de (tmmx − tmmn) mensual, equivalente a BIO2 de WorldClim.

In [ ]:
# --- Relieve (SRTM, 30 m)
dem = ee.Image('USGS/SRTMGL1_003')
altitud = dem.rename('altitud')

# --- Climatología reciente de temperatura (TerraClimate, ~4,6 km) con corrección altitudinal
terra = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
         .filterDate(*PERIODO_CLIMA)
         .map(lambda i: i.resample('bilinear')))
terra_proj = ee.Image(ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').first()).projection()

t_media_gruesa = terra.map(lambda i: i.select('tmmx').add(i.select('tmmn'))
                           .multiply(0.05)).mean()            # (tmmx + tmmn) / 2, escala 0,1
dem_grueso = (dem.reduceResolution(ee.Reducer.mean(), maxPixels=65535)
              .reproject(crs=terra_proj).resample('bilinear'))
t_media_fina = t_media_gruesa.add(dem.subtract(dem_grueso).multiply(GRADIENTE_TERMICO))

def amplitud_de(col):
    return col.map(lambda i: i.select('tmmx').subtract(i.select('tmmn')).multiply(0.1)).mean()

temperatura = t_media_fina.rename('temperatura')
amplitud = amplitud_de(terra).rename('amplitud')

pendiente = (ee.Terrain.slope(dem).multiply(3.141592653589793 / 180).tan()
             .multiply(100).rename('pendiente'))     # %

# --- Lluvia anual media (CHIRPS)
anios = ee.Number(ee.Date(PERIODO_CLIMA[1]).difference(ee.Date(PERIODO_CLIMA[0]), 'year'))
lluvia = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate(*PERIODO_CLIMA)
          .map(lambda img: img.resample('bilinear'))
          .sum().divide(anios).rename('lluvia'))     # mm/año

# --- Humedad relativa (ERA5-Land mensual, ~11 km)
era = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
       .filterDate(*PERIODO_CLIMA)
       .map(lambda img: img.resample('bilinear'))
       .mean())
humedad = humedad_relativa(era.select('temperature_2m').subtract(273.15),
                           era.select('dewpoint_temperature_2m').subtract(273.15)).rename('humedad')

# --- pH del suelo (OpenLandMap, 250 m) y cobertura (ESA WorldCover, 10 m)
ph = (ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02')
      .select('b0').multiply(0.1).rename('ph'))
cobertura = ee.ImageCollection('ESA/WorldCover/v200').first().rename('cobertura')

capas = (ee.Image.cat([altitud, pendiente, temperatura, amplitud, lluvia, humedad, ph])
         .clip(region))
print(capas.bandNames().getInfo())

## 6. Restricciones para la aptitud

In [ ]:
permitido = cobertura.remap(COBERTURAS_EXCLUIDAS, [0] * len(COBERTURAS_EXCLUIDAS), 1)
protegidas = (ee.FeatureCollection('WCMC/WDPA/current/polygons')
              .filterBounds(region))
if EXCLUIR_AREAS_PROTEGIDAS:
    permitido = permitido.And(ee.Image.constant(1).paint(protegidas, 0))
permitido = permitido.clip(region)

## 7. Aptitud para café

In [ ]:
aptitud = indice(APTITUD_CAFE, 'aptitud').updateMask(permitido)
aptitud_clase = clasificar(aptitud)

def capa_bordes(m):
    m.addLayer(municipios.style(color='#555555', fillColor='00000000', width=1), {}, 'Municipios')
    m.addLayer(cauca.style(color='black', fillColor='00000000', width=2), {}, 'Cauca')

fuera_apt = aptitud_clase.unmask(0).eq(0).selfMask().clip(region)

m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
m.centerObject(cauca, 8)
m.addLayer(fuera_apt, {'palette': [COLOR_FUERA]}, 'Excluido', True, 0.75)
m.addLayer(aptitud_clase, {'min': 1, 'max': 5, 'palette': PALETA_APT}, 'Aptitud café')
capa_bordes(m)
leyenda = dict(zip(ETIQ_APT, PALETA_APT)); leyenda['Excluido'] = COLOR_FUERA
m.add_legend(title='Aptitud para café', legend_dict=leyenda)
m

## 8. Área sembrada de café

Con `'fdp'` se usa el mapa de probabilidad de presencia de café de Forest Data Partnership (Google, 10 m). Es un producto comunitario en evolución: **compara la superficie que imprime esta celda con la cifra oficial de la Federación Nacional de Cafeteros para el Cauca** y ajusta `UMBRAL_PROB_CAFE` si hay mucha diferencia.

In [ ]:
def area_sembrada_cafe():
    """Máscara (1 = café) del área sembrada, según FUENTE_AREA_CAFE."""
    if FUENTE_AREA_CAFE == 'fdp':
        prob = (ee.ImageCollection('projects/forestdatapartnership/assets/coffee/model_2026a')
                .filterDate(f'{ANIO_CAFE}-01-01', f'{ANIO_CAFE + 1}-01-01')
                .mosaic().select('probability'))
        return prob.gte(UMBRAL_PROB_CAFE).selfMask().rename('cafe').clip(region)
    if FUENTE_AREA_CAFE == 'asset':
        poligonos = ee.FeatureCollection(ASSET_AREA_CAFE)
        return ee.Image().byte().paint(poligonos, 1).selfMask().rename('cafe').clip(region)
    raise ValueError("FUENTE_AREA_CAFE debe ser 'fdp' o 'asset' en este paso")

def hectareas(mascara, escala=30):
    res = (ee.Image.pixelArea().divide(10000).updateMask(mascara)
           .reduceRegion(ee.Reducer.sum(), region, escala, maxPixels=1e11, tileScale=8))
    return ee.Number(res.get('area'))

if FUENTE_AREA_CAFE == 'modelo':
    alt = capas.select('altitud')
    cafe = aptitud.gte(0.4).And(alt.gte(1000)).And(alt.lte(2200)).selfMask().rename('cafe')
else:
    cafe = area_sembrada_cafe()

print(f'Área de café ({FUENTE_AREA_CAFE}): {hectareas(cafe).getInfo():,.0f} ha')

m_cafe = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
m_cafe.centerObject(cauca, 8)
m_cafe.addLayer(cafe, {'palette': ['#6a3d1e']}, 'Café sembrado')
capa_bordes(m_cafe)
m_cafe

## 9. Riesgo de roya y de muerte descendente sobre el área sembrada

Las zonas sin café se muestran en gris. Como los lotes son pequeños, a escala departamental el mapa se ve como puntos: acerca el zoom a un municipio para verlo bien.

In [ ]:
roya = indice(RIESGO_ROYA, 'riesgo_roya').updateMask(cafe)
phoma = indice(RIESGO_PHOMA, 'riesgo_phoma').updateMask(cafe)
roya_clase = clasificar(roya)
phoma_clase = clasificar(phoma)

sin_cafe = cafe.unmask(0).eq(0).selfMask().clip(region)
vis_riesgo = {'min': 1, 'max': 5, 'palette': PALETA_RIESGO}

def mapa_riesgo(clase, titulo):
    m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
    m.centerObject(cauca, 8)
    m.addLayer(sin_cafe, {'palette': [COLOR_FUERA]}, 'Sin café', True, 0.75)
    m.addLayer(clase, vis_riesgo, titulo)
    capa_bordes(m)
    leyenda = dict(zip(ETIQ_RIESGO, PALETA_RIESGO)); leyenda['Sin café'] = COLOR_FUERA
    m.add_legend(title=titulo, legend_dict=leyenda)
    return m

In [ ]:
mapa_riesgo(roya_clase, 'Riesgo de roya')

In [ ]:
mapa_riesgo(phoma_clase, 'Riesgo de Phoma')

## 10. Diagnóstico: rango de las variables en el área sembrada

Si una variable tiene casi todo su rango por encima del límite superior de un criterio `sube` (o por debajo del inferior de un `baja`), ese criterio está saturado y no distingue entre sitios: ajusta sus umbrales en la sección 2.

In [ ]:
import pandas as pd

stats = capas.updateMask(cafe).reduceRegion(
    reducer=ee.Reducer.percentile([5, 25, 50, 75, 95]),
    geometry=region, scale=250, maxPixels=1e10, tileScale=4).getInfo()

filas = {}
for clave, valor in stats.items():
    variable, p = clave.rsplit('_', 1)
    filas.setdefault(variable, {})[p] = None if valor is None else round(valor, 1)
display(pd.DataFrame(filas).T[['p5', 'p25', 'p50', 'p75', 'p95']])

## 11. Superficie por clase (hectáreas)

In [ ]:
def area_por_clase(clase, etiquetas, escala):
    img = ee.Image.pixelArea().divide(10000).addBands(clase)
    res = img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='clase'),
        geometry=region, scale=escala, maxPixels=1e11, tileScale=8).getInfo()
    filas = [{'clase': etiquetas[int(g['clase']) - 1], 'hectareas': round(g['sum'])}
             for g in res['groups']]
    df = pd.DataFrame(filas)
    df['%'] = (100 * df['hectareas'] / df['hectareas'].sum()).round(1)
    return df

print('Aptitud para café (departamento)'); display(area_por_clase(aptitud_clase, ETIQ_APT, 100))
print('Riesgo de roya (área sembrada)');   display(area_por_clase(roya_clase, ETIQ_RIESGO, 30))
print('Riesgo de Phoma (área sembrada)');  display(area_por_clase(phoma_clase, ETIQ_RIESGO, 30))

## 12. Sensibilidad a los pesos: uno a la vez

Cada peso se sube y se baja en `SENS_CAMBIOS` (±20 %) y los demás se reescalan para que sigan sumando 1. La tabla muestra qué porcentaje del área cambia de clase y cuánto cambia el índice en promedio. Un criterio con mucho cambio es uno cuyo peso hay que justificar bien.

In [ ]:
import numpy as np

def perturbar(pesos, variable, cambio):
    p = dict(pesos)
    p[variable] = p[variable] * (1 + cambio)
    s = sum(p.values())
    return {k: v / s for k, v in p.items()}

INDICES = [('Aptitud', APTITUD_CAFE, permitido),
           ('Roya', RIESGO_ROYA, cafe),
           ('Phoma', RIESGO_PHOMA, cafe)]

bandas, escenarios = [], []
for nombre, crit, dominio in INDICES:
    base = indice(crit, 'base').updateMask(dominio)
    base_clase = clasificar(base)
    for variable in crit:
        for cambio in SENS_CAMBIOS:
            pesos = perturbar(pesos_base(crit), variable, cambio)
            esc = indice(crit, 'esc', pesos=pesos).updateMask(dominio)
            k = len(escenarios)
            bandas += [clasificar(esc).neq(base_clase).rename(f'c{k}'),
                       esc.subtract(base).abs().rename(f'd{k}')]
            escenarios.append((nombre, variable, cambio))

res = ee.Image.cat(bandas).reduceRegion(
    ee.Reducer.mean(), region, SENS_ESCALA, maxPixels=1e10, tileScale=8).getInfo()

sens = pd.DataFrame([{
    'índice': n, 'criterio': v, 'cambio del peso': f'{c:+.0%}',
    '% área que cambia de clase': round(100 * (res[f'c{k}'] or 0), 1),
    'cambio medio del índice': round(res[f'd{k}'] or 0, 4)}
    for k, (n, v, c) in enumerate(escenarios)])
display(sens)

## 13. Sensibilidad a los pesos: Monte Carlo

Se simulan `SENS_N_SIM` juegos de pesos al azar (distribución de Dirichlet centrada en los pesos base) y se mide, en cada píxel, la fracción de simulaciones que dan la misma clase que los pesos base. Un **acuerdo ≥ 0,8** indica que la clase es robusta a la elección de pesos.

In [ ]:
rng = np.random.default_rng(SEMILLA)

def monte_carlo(criterios, dominio):
    base_p = pesos_base(criterios)
    variables = list(base_p)
    alfa = np.array([base_p[v] for v in variables]) * SENS_CONCENTRACION
    simulados = rng.dirichlet(alfa, SENS_N_SIM)
    base_clase = clasificar(indice(criterios, 'base').updateMask(dominio))
    coincide = [clasificar(indice(criterios, 'sim', pesos=dict(zip(variables, w)))
                           .updateMask(dominio)).eq(base_clase)
                for w in simulados]
    acuerdo = ee.ImageCollection.fromImages(coincide).mean().rename('acuerdo')
    rango = pd.DataFrame(simulados, columns=variables).agg(['min', 'mean', 'max']).round(3)
    return acuerdo, rango

resumen, acuerdos = [], {}
for nombre, crit, dominio in INDICES:
    acuerdo, rango = monte_carlo(crit, dominio)
    acuerdos[nombre] = acuerdo
    r = (acuerdo.addBands(acuerdo.gte(0.8).rename('robusto'))
         .reduceRegion(ee.Reducer.mean(), region, SENS_ESCALA, maxPixels=1e10, tileScale=8)
         .getInfo())
    resumen.append({'índice': nombre, 'acuerdo medio': round(r['acuerdo'], 3),
                    '% área con acuerdo ≥ 0,8': round(100 * r['robusto'], 1)})
    print(f'Pesos simulados para {nombre}:'); display(rango)

display(pd.DataFrame(resumen))

In [ ]:
m_sens = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
m_sens.centerObject(cauca, 8)
vis_acuerdo = {'min': 0.5, 'max': 1, 'palette': ['#d7191c', '#ffffbf', '#1a9641']}
for nombre in ['Aptitud', 'Roya', 'Phoma']:
    m_sens.addLayer(acuerdos[nombre], vis_acuerdo, f'Acuerdo {nombre}', nombre == 'Roya')
capa_bordes(m_sens)
m_sens.add_colorbar(vis_acuerdo, label='Fracción de simulaciones con la misma clase')
m_sens

## 14. Exportar

### 14.1 Imágenes PNG (fondo transparente, Cauca recortado, municipios)

In [ ]:
import requests, zipfile
from IPython.display import Image, display
from google.colab import files

base_gris = ee.Image.constant(1).clip(region).visualize(palette=[COLOR_FUERA.lstrip('#')])
lineas_mpio = ee.Image().byte().paint(municipios, 1, 1).visualize(palette=['555555'])
borde_depto = ee.Image().byte().paint(cauca, 1, 3).visualize(palette=['000000'])

def guardar_png(clase, paleta, nombre):
    color = clase.visualize(min=1, max=5, palette=[c.lstrip('#') for c in paleta])
    mapa = ee.ImageCollection([base_gris, color, lineas_mpio, borde_depto]).mosaic()
    url = mapa.getThumbURL({'region': region.bounds(), 'dimensions': 2400,
                            'format': 'png', 'crs': 'EPSG:4326'})
    with open(nombre, 'wb') as f:
        f.write(requests.get(url).content)
    print(nombre); display(Image(nombre, width=450))

pngs = [('aptitud_cafe.png', aptitud_clase, PALETA_APT),
        ('riesgo_roya.png', roya_clase, PALETA_RIESGO),
        ('riesgo_phoma.png', phoma_clase, PALETA_RIESGO)]
for nombre, clase, paleta in pngs:
    guardar_png(clase, paleta, nombre)

with zipfile.ZipFile('mapas_cafe_cauca_png.zip', 'w') as z:
    for nombre, _, _ in pngs:
        z.write(nombre)
files.download('mapas_cafe_cauca_png.zip')

### 14.2 GeoTIFF coloreados para QGIS (descarga directa, 250 m)

Cada archivo trae la paleta y el fondo transparente. Valores: 1 a 5 = clases; 0 = sin dato. A 250 m los lotes de café muy pequeños pueden perderse; para más detalle usa 14.3.

In [ ]:
!pip install -q rasterio
import rasterio

origen = 'cauca_cafe_indices.tif'
salida = ee.Image.cat([aptitud, roya, phoma]).toFloat().unmask(-1).add(1)  # 0 = sin dato
geemap.ee_export_image(salida, filename=origen, scale=250, region=region, crs='EPSG:4326')

tifs = [('aptitud_cafe', PALETA_APT), ('riesgo_roya', PALETA_RIESGO), ('riesgo_phoma', PALETA_RIESGO)]
with rasterio.open(origen) as src:
    perfil = src.profile.copy()
    perfil.update(count=1, dtype='uint8', nodata=0, compress='lzw')
    for banda, (nombre, paleta) in enumerate(tifs, start=1):
        v = src.read(banda)
        ok = v >= 1
        x = v[ok] - 1
        clase = np.zeros(v.shape, dtype='uint8')
        clase[ok] = 1 + (x >= 0.2) + (x >= 0.4) + (x >= 0.6) + (x >= 0.8)
        colores = {0: (0, 0, 0, 0)}
        colores.update({i + 1: hex_a_rgba(c) for i, c in enumerate(paleta)})
        with rasterio.open(nombre + '.tif', 'w', **perfil) as dst:
            dst.write(clase, 1)
            dst.write_colormap(1, colores)
        print(nombre, '- sin dato:', round(100 * (~ok).mean()), '%')

with zipfile.ZipFile('mapas_cafe_cauca_tif.zip', 'w') as z:
    for nombre, _ in tifs:
        z.write(nombre + '.tif')
files.download('mapas_cafe_cauca_tif.zip')

### 14.3 Exportar a Google Drive a 30 m (opcional)

Necesita que la autenticación tenga permisos completos (no de solo lectura). El avance se ve en https://code.earthengine.google.com/tasks

In [ ]:
tarea = ee.batch.Export.image.toDrive(
    image=ee.Image.cat([aptitud, roya, phoma]).toFloat(),
    description='cauca_cafe_aptitud_riesgo_30m', folder='cauca_gee',
    region=region, scale=30, crs='EPSG:4326', maxPixels=1e11)
tarea.start()
print(tarea.status())